In [1]:
library(keras)
library(tensorflow)
library(tidyverse)
library(recipes)
set.seed(2) 

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.2     ✔ readr     2.1.4
✔ forcats   1.0.0     ✔ stringr   1.5.0
✔ ggplot2   3.4.2     ✔ tibble    3.2.1
✔ lubridate 1.9.2     ✔ tidyr     1.3.0
✔ purrr     1.0.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attachement du package : 'recipes'


L'objet suivant est masqué depuis 'package:stringr':

    fixed


L'objet suivant est masqué depuis 'package:stats':

    step




In [2]:
ConfusionMatrix <- function(y_pred, y_true) {
  Confusion_Mat <- table(y_true, y_pred)
  return(Confusion_Mat)
}
 
ConfusionDF <- function(y_pred, y_true) {
  Confusion_DF <- transform(as.data.frame(ConfusionMatrix(y_pred, y_true)),
                            y_true = as.character(y_true),
                            y_pred = as.character(y_pred),
                            Freq = as.integer(Freq))
  return(Confusion_DF)
}
 
Precision_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FP <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # it may happen that a label is never predicted (missing from y_pred) but exists in y_true
    # in this case ConfusionDF will not have these lines and thus the simplified code crashes
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]))
   
    # workaround:
    # i don't want to change ConfusionDF since i don't know if the current behaviour is a feature or a bug.
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
   
    tmp <- Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]
    FP[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Precision_micro <- sum(TP) / (sum(TP) + sum(FP))
  return(Precision_micro)
}
 
Recall_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FN <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # short version, comment out due to bug or feature of Confusion_DF
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]))
   
    # workaround:
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
 
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]
    FN[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Recall_micro <- sum(TP) / (sum(TP) + sum(FN))
  return(Recall_micro)
}
 
F1_Score_micro <- function(y_true, y_pred, labels = NULL) {
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred)) # possible problems if labels are missing from y_*
  Precision <- Precision_micro(y_true, y_pred, labels)
  Recall <- Recall_micro(y_true, y_pred, labels)
  F1_Score_micro <- 2 * (Precision * Recall) / (Precision + Recall)
  return(F1_Score_micro)
}

In [3]:
# data<-read.csv("train_values.csv",stringsAsFactors = T)
# data_labels<-read.csv("train_labels.csv",stringsAsFactors = T)
# datam<-merge(data,data_labels,by=c('building_id','building_id'))

In [6]:
# target_variable<-match('damage_grade', colnames(datam))
# id_loc_1 <- match('geo_level_1_id', colnames(datam))
# id_loc_2 <- match('geo_level_2_id', colnames(datam))
# id_loc_3 <- match('geo_level_3_id', colnames(datam))
# id_loc_1_vals<-sort(unique(datam[,id_loc_1]))
# id_loc_2_vals<-sort(unique(datam[,id_loc_2]))
# id_loc_3_vals<-sort(unique(datam[,id_loc_3]))
# id_loc_1_df <- data.frame(matrix(ncol = 2, nrow = length(id_loc_1_vals)))
# colnames(id_loc_1_df)<-c('id','mean')
# id_loc_2_df <- data.frame(matrix(ncol = 2, nrow = length(id_loc_2_vals)))
# colnames(id_loc_2_df)<-c('id','mean')
# id_loc_3_df <- data.frame(matrix(ncol = 2, nrow = length(id_loc_3_vals)))
# colnames(id_loc_3_df)<-c('id','mean')

# for (i in 1:length(id_loc_1_vals)){
#     id_loc_1_df[i,1]<-id_loc_1_vals[i]
#     id_loc_1_df[i,2]<-mean(filter(datam,geo_level_1_id == id_loc_1_vals[i])[,target_variable])
# }
# for (i in 1:length(id_loc_2_vals)){
#     id_loc_2_df[i,1]<-id_loc_2_vals[i]
#     id_loc_2_df[i,2]<-mean(filter(datam,geo_level_2_id == id_loc_2_vals[i])[,target_variable])
# }
# for (i in 1:length(id_loc_3_vals)){
#     id_loc_3_df[i,1]<-id_loc_3_vals[i]
#     id_loc_3_df[i,2]<-mean(filter(datam,geo_level_3_id == id_loc_3_vals[i])[,target_variable])
# }

In [3]:
# datam<-add_column(datam,geo_level_1_mean_damage = NA,.after='geo_level_1_id')
# datam<-add_column(datam,geo_level_2_mean_damage = NA,.after='geo_level_2_id')
# datam<-add_column(datam,geo_level_3_mean_damage = NA,.after='geo_level_3_id')
# id_loc_1 <- match('geo_level_1_id', colnames(datam))
# id_loc_2 <- match('geo_level_2_id', colnames(datam))
# id_loc_3 <- match('geo_level_3_id', colnames(datam))
# for (i in 1:nrow(datam)){
#     datam[i,id_loc_1+1]<-filter(id_loc_1_df,id == datam[i,id_loc_1])[1,2]
#     datam[i,id_loc_2+1]<-filter(id_loc_2_df,id == datam[i,id_loc_2])[1,2]
#     datam[i,id_loc_3+1]<-filter(id_loc_3_df,id == datam[i,id_loc_3])[1,2]
# }
#write.csv(datam,'data_geoprocessed.csv')


datam<-read.csv("data_geoprocessed.csv",stringsAsFactors = T)
id_variable <- match('building_id', colnames(datam))
target_variables<-match('damage_grade', colnames(datam))

#DO NOT FORGET processing for test values which zones could not be in training set


In [4]:
summary(datam)

       X           building_id      geo_level_1_id geo_level_1_mean_damage
 Min.   :     1   Min.   :      4   Min.   : 0.0   Min.   :1.731          
 1st Qu.: 65151   1st Qu.: 261190   1st Qu.: 7.0   1st Qu.:2.026          
 Median :130301   Median : 525757   Median :12.0   Median :2.172          
 Mean   :130301   Mean   : 525676   Mean   :13.9   Mean   :2.238          
 3rd Qu.:195451   3rd Qu.: 789762   3rd Qu.:21.0   3rd Qu.:2.446          
 Max.   :260601   Max.   :1052934   Max.   :30.0   Max.   :2.794          
                                                                          
 geo_level_2_id   geo_level_2_mean_damage geo_level_3_id 
 Min.   :   0.0   Min.   :1.000           Min.   :    0  
 1st Qu.: 350.0   1st Qu.:2.018           1st Qu.: 3073  
 Median : 702.0   Median :2.210           Median : 6270  
 Mean   : 701.1   Mean   :2.238           Mean   : 6258  
 3rd Qu.:1050.0   3rd Qu.:2.480           3rd Qu.: 9412  
 Max.   :1427.0   Max.   :3.000           Max.   :12

I try without removing nzv values. We will try it later to see if it improves predict.

In [4]:
dataNN <- recipe(damage_grade ~ ., datam) %>%
  step_nzv(everything(), -damage_grade) %>%
  step_normalize(count_floors_pre_eq, age, area_percentage, height_percentage) %>%
  step_num2factor(damage_grade,levels=c('1','2','3')) %>%
  step_dummy(all_nominal(), one_hot = TRUE) %>%
  step_rm(geo_level_1_id,geo_level_2_id,geo_level_3_id,X,building_id) %>%
  prep(log_changes=TRUE) %>%
  bake(new_data = NULL)

step_nzv (nzv_Y2hMG): 
 removed (16): plan_configuration, has_superstructure_stone_flag, ...

step_normalize (normalize_c1pFX): same number of columns

step_num2factor (num2factor_8OXka): same number of columns

step_dummy (dummy_hqCql): 
 new (27): land_surface_condition_n, land_surface_condition_o, ...
 removed (7): land_surface_condition, foundation_type, roof_type, ...

step_rm (rm_uoffF): 
 removed (5): X, building_id, geo_level_1_id, geo_level_2_id, ...



In [5]:
nrows<-nrow(dataNN)
split <- floor(nrows*0.8)
dataNN_idx <- sample(1:nrows)
train_data <- dataNN[dataNN_idx[1:split],]
test_data <- dataNN[dataNN_idx[(split+1):nrows],]
target_indices<-which(grepl('damage_grade',colnames(dataNN)))

In [6]:
normalizer<-layer_normalization(axis = -1L)  %>%  adapt(as.matrix(train_data[,-target_indices]))


We use categorical_crossentropy as loss function since this corresponds to our multioutput classification program

In [7]:
neuralmodel <- keras_model_sequential() %>% 
normalizer  %>% 
layer_dense(64, activation = 'relu') %>%
layer_dense(64, activation = 'relu') %>%
layer_dense(3,activation='softmax')

neuralmodel %>% compile(
    loss = 'categorical_crossentropy',
    optimizer = optimizer_adam(0.001),
    metrics=c('accuracy')
  )
neuralmodel

Model: "sequential"
________________________________________________________________________________
 Layer (type)                  Output Shape               Param #    Trainable  
 normalization (Normalization)  (None, 40)                81         Y          
 dense_2 (Dense)               (None, 64)                 2624       Y          
 dense_1 (Dense)               (None, 64)                 4160       Y          
 dense (Dense)                 (None, 3)                  195        Y          
Total params: 7,060
Trainable params: 6,979
Non-trainable params: 81
________________________________________________________________________________

In [8]:
yhat <- neuralmodel %>% fit(
  as.matrix(train_data[,-target_indices]),
  as.matrix(train_data[,target_indices]),
  validation_split = 0.2,
  verbose = 0,
  epochs = 100
)
yhat


Final epoch (plot to see history):
        loss: 0.5243
    accuracy: 0.7626
    val_loss: 0.5873
val_accuracy: 0.7429 

In [9]:
yhatt<-neuralmodel  %>% evaluate(
    as.matrix(test_data[,-target_indices]),
    as.matrix(test_data[,target_indices]),
)
yhatt

loss  accuracy 
0.5907099 0.7423495

In [10]:
yhattt <- predict(neuralmodel, as.matrix(test_data[-target_indices]))
yhattt

4.211471e-03,0.966300011,0.029488534
1.444790e-01,0.825844586,0.029676368
1.429237e-02,0.936121285,0.049586348
3.976784e-05,0.807366252,0.192594007
3.554568e-04,0.529320061,0.470324457
1.829507e-02,0.494922698,0.486782223
9.429686e-01,0.036002219,0.021029180
6.559361e-01,0.342370331,0.001693565
7.327980e-03,0.823945165,0.168726906
3.416021e-04,0.689309001,0.310349435
5.414045e-03,0.776766717,0.217819288


In [11]:
yhat_f<-data.frame(matrix(0,ncol = 1, nrow = nrow(yhattt)))
y_f<-data.frame(matrix(0,ncol = 1, nrow = nrow(yhattt)))
for (i in 1:nrow(yhattt)){
    yhat_f[i,]<-which.max(yhattt[i,])
    y_f[i,]<-which.max(test_data[i,target_indices])  
}
colnames(yhat_f)<-'damage_grade'
colnames(y_f)<-'damage_grade'

In [12]:
print(paste('F1 Score Micro: ',F1_Score_micro(as.factor(y_f[,]),as.factor(yhat_f[,]))))

[1] "F1 Score Micro:  0.742349532817866"


Predicting the testing set:

In [17]:
# data_t<-read.csv("test_values.csv",stringsAsFactors = T)

# data_t<-add_column(data_t,geo_level_1_mean_damage = NA,.after='geo_level_1_id')
# data_t<-add_column(data_t,geo_level_2_mean_damage = NA,.after='geo_level_2_id')
# data_t<-add_column(data_t,geo_level_3_mean_damage = NA,.after='geo_level_3_id')
# id_loc_1 <- match('geo_level_1_id', colnames(data_t))
# id_loc_2 <- match('geo_level_2_id', colnames(data_t))
# id_loc_3 <- match('geo_level_3_id', colnames(data_t))
# for (i in 1:nrow(data_t)){
#     data_t[i,id_loc_1+1]<-filter(id_loc_1_df,id == data_t[i,id_loc_1])[1,2]
#     data_t[i,id_loc_2+1]<-filter(id_loc_2_df,id == data_t[i,id_loc_2])[1,2]
#     data_t[i,id_loc_3+1]<-filter(id_loc_3_df,id == data_t[i,id_loc_3])[1,2]
# }
# write.csv(data_t,'testdata_geoprocessed.csv')

data_t<-read.csv("testdata_geoprocessed.csv",stringsAsFactors = T)

In [42]:
dataNN_test <- recipe( ~ ., data_t) %>%
  #step_nzv(everything(), ) %>%
  step_normalize(count_floors_pre_eq, age, area_percentage, height_percentage,building_id) %>%
  step_dummy(all_nominal(), one_hot = TRUE) %>%
  step_rm(geo_level_1_id,geo_level_2_id,geo_level_3_id,X,building_id) %>%
  prep(log_changes=TRUE) %>%
  bake(new_data = NULL)

test_predict <- predict(neuralmodel, as.matrix(dataNN_test[,-1]))
test_predict_f<-data.frame(matrix(0,ncol = 2, nrow = nrow(test_predict)))
colnames(test_predict_f)<-c('building_id','damage_grade')
test_predict<-replace_na(test_predict,0)

for (i in 1:nrow(test_predict)){
    test_predict_f[i,1]<-data_t[i,2]
    test_predict_f[i,2]<-which.max(test_predict[i,])
}
write.csv(test_predict_f,'prediction_temp_NN.csv',col.names=TRUE,row.names=FALSE)